# Day three — Species distribution modeling

---

-> summary: we have records of where species have been observed, but those points are not yet an explanation. Where is the species found? Where is it absent, or simply not recorded? What environmental conditions might explain the pattern? Once we have an idea of what determines the presence, absence, or even abundance of a species, we can go beyond observation, and model the species’ distribution. With such models, we can predict where we have no data, identify key regions for conservation, and even model what could happen if the environment were to change.

- PART I: From observations to ecological questions.
- PART II: Identifying environmental predictors: climate, land use, and habitat suitability.
- PART III: Species distribution modeling.
- PART IV: How to be a good scientist: tips, tricks, and pitfalls.


## *PART I: From observations to ecological questions*

## 📋 Planning note (instructor-facing — delete before making the student `day3.ipynb`)

**Target species: Philippine tarsier (*Carlito syrichta*).** Endemic, IUCN Near Threatened, and — unlike
the earlier flying-fox / eagle candidates — genuinely climate/habitat-restricted rather than either
ubiquitous (*Macaca fascicularis*) or archipelago-wide (*Pteropus vampyrus*), which should give the
model a real signal to find. Lives inside the broader `gbif_mammals_philippines.csv` compilation
(no dedicated single-species pull like Day 2's flying fox/eagle): 408 records within the PH bbox, 312
unique ~100 m locations, spanning 6 islands / 27 provinces. Two data-quality findings from checking this
before committing (see below — both become teaching material, not just cleanup):
- A dense cluster on Bohol (~124°E, 9.8°N) coincides with the Tarsier Sanctuary / tourism sites — likely
  captive or semi-captive individuals, not independent wild observations.
- **6 records fall well outside the species' known native range** (the Mindanao faunal region: Mindanao,
  Bohol, Leyte, Samar, and nearby islands) — one near 15°N in central Luzon, three around
  121.7–122°E/12.7–12.9°N (Mindoro/Romblon). Almost certainly zoo/captive records mislabeled as wild;
  these get filtered out in Part I rather than fed to the model.

**Data inventory & feasibility (updated):**

| Data | Resolution | Extent | Status |
|---|---|---|---|
| Tarsier occurrences | point | Philippines-wide | ✅ have — filter `gbif_mammals_philippines.csv` to `species == "Carlito syrichta"` |
| Köppen-Geiger classification | 1 km (~0.0083°) | Philippines-wide | ✅ have (`kg_philippines_present.nc` + `..._future_ssp460.nc`) |
| WorldClim BIO5 (max temp of warmest month) & BIO14 (precip of driest month) | 30 arc-sec (~1 km) | Philippines-wide | ✅ have (`worldclim_bio_philippines_1970_2000.nc`) — 1970–2000 baseline, the only period WorldClim offers at this resolution |
| Elevation (Copernicus DEM GLO-30, aggregated) | 1 km | Philippines-wide | ✅ have (`dem_philippines_1km.nc`) — native 30 m block-averaged onto the same grid, no raw 30 m file kept |
| Land-sea mask | 0.25° / 0.1° / 1 km | Philippines-wide | ✅ have (3 resolutions) |
| Future climate (ERA5-derived, SSP4-6.0) | 0.1° (~11 km) | Philippines-wide | ✅ have (`climate_philippines_2071_2099_ssp460_mean.nc`) — **different lineage and coarser than the 1 km present-day layers above; see capstone note in Part III** |
| Forest cover (Hansen) / satellite imagery | 30 m / ~100–200 m | **Negros only** | ⚠️ partial — unchanged from Day 2, still an optional regional stretch only |

**Resolution strategy (upgraded from the original plan):** KG, WorldClim BIO5/BIO14, the land-sea mask,
and the DEM are all now aligned to the *same* ~1 km national grid (2040 × 1620 cells,
lon 113.5–127.0°, lat 4.5–21.5°) — see `behind-the-scenes/day3_bts.ipynb` §1–2. That's a real upgrade
over the original plan (which would have capped the national model at the 0.1° ERA5 climatology): the
core model now runs at ~1 km nationally without needing the Negros-only Hansen/Sentinel layers at all.

**Watch out for: the bbox is a rectangle, not the Philippines.** Because it's a plain lon/lat box, the
grid's southwest corner (roughly lon < 119°, lat < 7.5°) includes a real slice of northern Borneo
(Sabah) — confirmed by the DEM's max elevation (~3900 m) matching Mt. Kinabalu, not any Philippine peak.
Presence points are unaffected (GBIF's own `country=PH` filter already restricts those), but **background
/ pseudo-absence sampling must exclude that corner explicitly**, or the model quietly learns from Bornean
climate. No new dependency needed (no shapefile/geopandas) — a hand-specified sub-box exclusion is
sufficient and stays in the workshop's "homemade" style. This doubles as a concrete Part IV pitfall.

**Scope note:** Day 3 should end up *shorter* than Day 2, so Parts II–III stay lean. Loading three
pre-aligned, ready-to-use national layers (instead of deriving bioclim-style summaries from monthly
ERA5 grids by hand) actually helps here — that derivation exercise is dropped from Day 3 since it's no
longer how the core predictors are produced (it's already a Day 1 skill; no need to repeat it). This is
also the first time students see a real fitted statistical model rather than a hand-coded rule tree
(Day 1/2's "homemade classifiers") — a deliberate step up, not a break in style. Part IV should stay short
discussion, not new heavy code.

In [ ]:
# 1
import os
if not os.path.exists("/content/USLS-Workshop"):
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop
%cd /content/USLS-Workshop
!pip install -q .

#from google.colab import drive
#drive.mount('/content/drive')

**Plan.** Goal: get students to look critically at presence-only occurrence data *before* jumping to
modeling — and this time the cleaning isn't hypothetical, it's two real findings worth walking through.

1. Filter `gbif_mammals_philippines.csv` (Day 2's broader mammal compilation) down to
   `species == "Carlito syrichta"` and the PH bbox — unlike the flying fox/eagle, the tarsier doesn't
   have its own dedicated pre-cleaned file, so this filtering step is new.
2. Plot presence points over a Philippines outline (reuse the land-sea-mask contour pattern from Day 1/2
   maps). Prompt: does the species look randomly scattered, or clustered? Two things should jump out:
   - A dense cluster on Bohol — home to the Tarsier Sanctuary and other tourist-facing sites. Discuss:
     is this real ecological signal, or repeated observation of a small number of captive/habituated
     individuals?
   - A handful of points far outside the species' known native range (the Mindanao faunal region —
     Mindanao, Bohol, Leyte, Samar, and nearby islands): one near central Luzon, a few around
     Mindoro/Romblon. Almost certainly zoo records mislabeled as wild.
3. Have students filter out the out-of-range points (a simple lat/lon rule is enough — no need for
   anything fancier) before moving on. This is the "clean, model-ready" presence set. The Bohol cluster
   is *not* removed (it's still real habitat), but flag it as a caveat to revisit in Part IV.
4. Discussion, pushing Day 2's sampling-bias theme specifically toward what it means *for modeling*:
   - GBIF gives us presences, not absences — an unrecorded area might be truly unsuitable, or just
     unsurveyed (near no roads/observers). We can't tell the difference from the points alone.
   - Introduce **background points / pseudo-absences** conceptually here (needed before Part III can fit
     anything): without real absences, we approximate them by sampling random points across the study
     area.
5. Land on the specific question Parts II-III will answer: *"Which combination of temperature,
   precipitation, and elevation is associated with where Philippine tarsiers have been recorded, and can
   that predict suitability elsewhere in the country?"*


   ### Load and filter occurrence records

Unlike the flying fox/eagle in Day 2, the tarsier doesn't have its own dedicated pre-cleaned file —
pull it out of the broader `gbif_mammals_philippines.csv` compilation from Day 2 Part II (all
present-day terrestrial mammal records for the Philippines).

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from workshop_utils import PROCESSED_DIR

mammals = pd.read_csv("../data/processed/day2/gbif_mammals_philippines.csv", low_memory=False)

tarsier_raw = mammals[mammals['species'] == 'Carlito syrichta'].dropna(
    subset=['decimalLongitude', 'decimalLatitude'])
tarsier_raw = tarsier_raw[
    tarsier_raw['decimalLongitude'].between(113.5, 127.0) &
    tarsier_raw['decimalLatitude'].between(4.5, 21.5)
]
print(f'Raw records: {len(tarsier_raw)}')
tarsier_raw.head()

### Look at the raw points before doing anything else

Plot them over a Philippines outline (same land-sea-mask contour trick from Day 1/2). Does the
species look randomly scattered, or clustered?

In [ ]:
lsm = xr.open_dataset(PROCESSED_DIR / 'land-sea-mask_0p00833333.nc')
land = lsm['land_sea_mask'].values > 0.5
LSM_LAT, LSM_LON = lsm['lat'].values, lsm['lon'].values

fig, ax = plt.subplots(figsize=(7, 9))
ax.contourf(LSM_LON, LSM_LAT, land, levels=[0.5, 1], colors=['#d9d9d9'])
ax.scatter(tarsier_raw['decimalLongitude'], tarsier_raw['decimalLatitude'], s=10, color='crimson')

ax.set_xlim(LSM_LON.min(), LSM_LON.max())
ax.set_ylim(LSM_LAT.min(), LSM_LAT.max())
ax.set_aspect('equal')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Philippine tarsier — {len(tarsier_raw)} raw GBIF records')

Two things should jump out:
- A dense cluster on Bohol (~124°E, 9.8°N) — home to the Tarsier Sanctuary and other tourist-facing
  sites. Is this real ecological signal, or repeated observation of a small number of
  captive/habituated individuals?
- A handful of points far outside the species' known native range (the Mindanao faunal region —
  Mindanao, Bohol, Leyte, Samar, and nearby islands): one near central Luzon, a few around
  Mindoro/Romblon. Almost certainly zoo records mislabeled as wild.

In [ ]:
# Native range = the Mindanao faunal region (Mindanao, Bohol, Leyte, Samar, nearby islands). The
# handful of points well outside it (central Luzon; Mindoro/Romblon) are almost certainly captive/zoo
# records mislabeled as wild -- drop them. The Bohol cluster is NOT removed here (it's still real
# habitat) -- flagged as a sampling-bias caveat to revisit in Part IV.
out_of_range = (
    (tarsier_raw['decimalLatitude'] > 13.5) |
    ((tarsier_raw['decimalLongitude'] < 122.5) & (tarsier_raw['decimalLatitude'] > 11.5))
)
print(f'Dropping {int(out_of_range.sum())} likely captive/out-of-range records')
tarsier = tarsier_raw[~out_of_range].copy()

# Remove spatial duplicates rounded to 0.01° (~1 km) -- the model will run on a 1 km grid anyway, so
# records that would land on the same grid cell carry no extra information for it
tarsier['lon_r'] = tarsier['decimalLongitude'].round(2)
tarsier['lat_r'] = tarsier['decimalLatitude'].round(2)
tarsier = tarsier.drop_duplicates(subset=['lon_r', 'lat_r'])
tarsier = tarsier.rename(columns={'decimalLongitude': 'lon', 'decimalLatitude': 'lat'})[['lon', 'lat']]

print(f'Clean presence set: {len(tarsier)} points')
tarsier.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.contourf(LSM_LON, LSM_LAT, land, levels=[0.5, 1], colors=['#d9d9d9'])

ax.scatter(tarsier['lon'], tarsier['lat'], s=10, color='crimson', label=f'Kept (n={len(tarsier)})')
dropped = tarsier_raw[out_of_range]
ax.scatter(dropped['decimalLongitude'], dropped['decimalLatitude'], s=50, facecolor='none',
           edgecolor='orange', linewidths=1.5, label=f'Dropped, out of range (n={len(dropped)})')

ax.set_xlim(LSM_LON.min(), LSM_LON.max())
ax.set_ylim(LSM_LAT.min(), LSM_LAT.max())
ax.set_aspect('equal')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Philippine tarsier — cleaned presence set')
ax.legend(loc='lower left', frameon=False)

### Presence-only data isn't the whole picture

GBIF gives us presences, not absences — an unrecorded area might be truly unsuitable, or just
unsurveyed (no roads, no observers nearby). We can't tell the difference from the points alone. To fit
a model in Part III we need something to contrast presences against: **background points**
(pseudo-absences). Without real absence surveys, we approximate them by sampling random points across
the study area — Part II builds these.